# Chapter 18 &mdash; Lambdas from a Programmer's Perspective

**Concept 2 of the Chapter 18 decomposition:** *Lambdas from a Programmer's Perspective: Functions Without Names*

A way to describe functions without redundantly attaching names to them.

---

*Run on Colab:* [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter18/Concept-Lambdas-For-Programmers/Concept-Lambdas-For-Programmers.ipynb)
*Or run locally from inside a Jove checkout.*

## 0. Setup

In [ ]:
# Run me first.  Works on Colab and on a local Jove checkout.
import os, subprocess, sys

def _git(*a):
    r = subprocess.run(('git',) + a, capture_output=True, text=True)
    return r.stdout.strip() if r.returncode == 0 else ''

REPO = 'https://github.com/ganeshutah/Jove'
try:                       # ---- Colab: clone once, pull thereafter ----
    import google.colab
    was = _git('-C', 'Jove', 'rev-parse', '--short', 'HEAD') if os.path.isdir('Jove') else ''
    if os.path.isdir('Jove') and not was:
        print('Jove: WARNING ./Jove exists but is not a git checkout -- left as is')
    elif was:
        _git('-C', 'Jove', 'pull', '-q', '--ff-only')
        now = _git('-C', 'Jove', 'rev-parse', '--short', 'HEAD')
        if now and now != was:
            print('Jove: PULLED  %s -> %s' % (was, now))
            print(_git('-C', 'Jove', 'log', '--oneline', was + '..' + now))
        else:
            print('Jove: PULLED  already current at %s' % (now or was))
    else:
        _git('clone', '-q', REPO, 'Jove')
        print('Jove: CLONED  at %s' % (_git('-C', 'Jove', 'rev-parse',
                                             '--short', 'HEAD') or '?'))
    JOVE = 'Jove'
except ImportError:        # ---- local: the checkout above Chapter<N>/ ----
    JOVE = next((p for p in ('../..', '../../..', '..', '.')
                 if os.path.isdir(os.path.join(p, 'jove'))), '../..')
    print('Jove: LOCAL   checkout at %s'
          % (_git('-C', JOVE, 'rev-parse', '--short', 'HEAD') or '?'))
sys.path.insert(0, JOVE)

# A session can already hold an OLDER jove in sys.modules.  The pull above
# updates the files on disk, but `import` would hand back the cached module --
# so a fixed library still behaves like the broken one.  Drop them first.
for _m in [k for k in list(sys.modules) if k == 'jove' or k.startswith('jove.')]:
    del sys.modules[_m]

from jove.Def_md2mc      import *
from jove.DotBashers     import *
from jove.Def_DFA        import *
from jove.Def_NFA        import *

import jove; print('Jove loaded from', list(jove.__path__)[0])

## 1. The idea


A programmer's summary: a lambda is a **function value without a name**.

```python
def add1(x): return x + 1     # a named function
add1 = lambda x: x + 1        # the same function, named afterwards
(lambda x: x + 1)(5)          # used without ever being named
```

Naming is often **redundant**: a comparator passed to `sort`, a callback, a one-line
transform. Forcing a name on those adds noise and puts the definition far from the use.

Three ideas come along with lambdas and matter more than the syntax:

* **first-class functions** &mdash; pass them, return them, store them;
* **closures** &mdash; a lambda captures the variables in scope where it was written;
* **currying** &mdash; a two-argument function *is* a one-argument function returning a
  one-argument function, which is exactly how the $\lambda$-calculus does everything.

## 2. Definitions

### Named versus anonymous

In [ ]:
def add1_named(x): return x + 1
add1_lambda = lambda x: x + 1

### Closures and currying

In [ ]:
def make_adder(n):
    return lambda x: x + n          # captures n -- a CLOSURE

curried_add = lambda a: lambda b: a + b
uncurried   = lambda a, b: a + b

def curry2(f):   return lambda a: lambda b: f(a, b)
def uncurry2(f): return lambda a, b: f(a)(b)

<!-- nav-strip -->

---

&larr;&nbsp;[Ch18&nbsp;1.&nbsp;The History of Lambda Calculus, and its Independence from Turing's Work](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter18/Concept-History-Of-Lambda/Concept-History-Of-Lambda.ipynb) &nbsp;&middot;&nbsp; [**Chapter 18** index](https://github.com/ganeshutah/Jove/blob/master/Chapter18/README.md) &nbsp;&middot;&nbsp; [Ch18&nbsp;3.&nbsp;Syntax and the Three Reduction Rules: Alpha, Beta, Eta](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter18/Concept-Syntax-And-Reduction-Rules/Concept-Syntax-And-Reduction-Rules.ipynb)&nbsp;&rarr;

---

## 3. Tests

The same function, two ways to write it.

In [ ]:
print("named  :", add1_named(5))
print("lambda :", add1_lambda(5))
print("inline :", (lambda x: x + 1)(5))
assert add1_named(5) == add1_lambda(5) == (lambda x: x + 1)(5) == 6

**Where anonymity pays:** the function is used once, at the point of definition.

In [ ]:
pairs = [('pear', 3), ('apple', 10), ('fig', 1)]
print("by count     :", sorted(pairs, key=lambda p: p[1]))
print("by name len  :", sorted(pairs, key=lambda p: len(p[0])))
print("\nNaming those two comparators would move them away from the sort.")

**Closures:** the lambda carries its environment.

In [ ]:
add5, add100 = make_adder(5), make_adder(100)
print("add5(1)   =", add5(1))
print("add100(1) =", add100(1))
assert add5(1) == 6 and add100(1) == 101
print("\nTwo different functions from one definition -- n was captured.")

The classic closure trap, and its fix.

In [ ]:
bad  = [lambda: i for i in range(3)]            # all capture the same i
good = [(lambda k: lambda: k)(i) for i in range(3)]
print("late binding :", [f() for f in bad])
print("captured now :", [f() for f in good])
assert [f() for f in bad] == [2, 2, 2]
assert [f() for f in good] == [0, 1, 2]
print("\nThe lambda captures the VARIABLE, not its value at definition time.")

**Currying:** the calculus has only one-argument functions.

In [ ]:
print("curried   :", curried_add(3)(4))
print("uncurried :", uncurried(3, 4))
assert curried_add(3)(4) == uncurried(3, 4) == 7
f = curry2(lambda a, b: a * b)
g = uncurry2(f)
assert f(6)(7) == g(6, 7) == 42
print("\ncurry2 and uncurry2 are inverses -- so n-argument functions are")
print("syntactic sugar, and the calculus loses nothing by omitting them.")

Partial application falls out for free.

In [ ]:
double = curried_add(0)
times  = lambda a: lambda b: a * b
triple = times(3)
print("triple(7) =", triple(7))
assert triple(7) == 21
print("\nApply one argument, keep the function.  That is all currying is.")

## 4. Exercises


1. Write `compose = lambda f: lambda g: lambda x: ...` and test it.
2. Why does Python's `lambda` allow only a single expression?
3. Curry a three-argument function. How many lambdas?

In [ ]:
# Your work for the exercises above.

## 5. Where next

In [ ]:
# Previous / next, and a search box for every concept.
# Type a chapter (Chapter7, ch7) or words from a title (pumping, subset).
#
# Following a link opens a NEW Colab runtime. To pull another concept's
# definitions into THIS session instead:  load_here('Chapter7/Concept-...')
import os, sys
try:                       # usually already done by the Setup cell
    import jove
except ModuleNotFoundError:
    _p = next((p for p in ('Jove', '../..', '../../..', '..', '.')
               if os.path.isdir(os.path.join(p, 'jove'))), None)
    if _p:
        sys.path.insert(0, _p)
try:
    from jove.Nav import nav, load_here
    nav(here='Chapter18/Concept-Lambdas-For-Programmers')
except ModuleNotFoundError:
    print('Jove is not on the path yet.')
    print('Run the Setup cell at the top of this notebook, then re-run this one.')